# 06 · Do Rossby waves emerge? A LAWFUL-style probe

Following *LAWFUL* (Chen et al. 2026): pair the model with a physics law and test whether its response to controlled perturbations follows the law.

* **Law:** barotropic Rossby waves on a zonal flow U, with c − U = −β / (k² + l²). Long waves drift westward relative to the flow, short ones less.
* **Perturbation:** plant a small, geostrophically balanced, non-divergent wave of zonal wavenumber s on real states.
* **Bridge:** measure the phase speed of the planted wave in the model's one-step response.
* **Score:** M = 1 − Σ(c_model − c_theory)² / Σ(c_theory − mean)², where 1 is perfect agreement with the law.

The model was never told about β. Latitude reaches it only through the position embedding.

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
try:
    import vit_interp
except ImportError:  # not installed: use this checkout, $VIT_INTERP_REPO, or the shared one
    for repo in (Path.cwd().parent, Path(os.environ.get("VIT_INTERP_REPO", "/home/ac.ngmahfouz/vit-interp"))):
        if (repo / "vit_interp").is_dir():
            sys.path.insert(0, str(repo)); break
import numpy as np, torch, matplotlib.pyplot as plt
from vit_interp import playground as pg
em, meta = pg.load()      # or pg.load("/path/to/checkpoint.pt"); $PLAYGROUND_CKPT also works
FAST = pg.FAST            # PLAYGROUND_FAST=1 -> small settings (CPU tests)

In [ ]:
from vit_interp import rossby as R
lat, lon = np.asarray(meta["lat"]), np.asarray(meta["lon"])
X = torch.cat([pg.state(t) for t in (["2010-01-15"] if FAST else ["2010-01-05", "2010-01-15", "2010-01-25", "2010-02-05"])])
rows, M = R.sweep(em, X, lat, lon, R.Wave(phi0=45, sigma=10, v_amp=1.0, levels=(2, 3, 4, 5, 6)),
                  ss=range(2, 8) if FAST else range(2, 15), level=3)
import pandas as pd
df = pd.DataFrame(rows); print("consistency M on the intrinsic speed:", round(M, 3)); df.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(df.s, df.c_model, "o-", label="ViT"); ax[0].plot(df.s, df.c_theory, "s--", label="Rossby theory")
ax[0].axhline(df.U_bar[0], color="gray", lw=0.5, label="Ū (advection only)"); ax[0].set_xlabel("zonal wavenumber s"); ax[0].set_ylabel("phase speed [m/s]"); ax[0].legend()
ax[1].plot(df.s, df.growth_per_day, "o-"); ax[1].set_xlabel("s"); ax[1].set_ylabel("growth [1/day]"); ax[1].set_title("does the wave grow or decay?");

M can be very negative even when the model has the right **shape**: Rossby-like dispersion with the wrong magnitude or an offset. A linear fit of the model's intrinsic speed against theory separates the two. Slope 1, intercept 0 and r = 1 would be the law exactly; r near 1 with slope ≠ 1 means the right physics, mis-scaled.

In [ ]:
a, b = np.polyfit(df.intrinsic_theory, df.intrinsic_model, 1)
r = np.corrcoef(df.intrinsic_theory, df.intrinsic_model)[0, 1]
print(f"intrinsic speed: model = {a:.2f} x theory {b:+.2f} m/s, r = {r:.3f}")

## Which components carry it?
Sufficiency: route the perturbation only through a set of components. Necessity: route it through everything else. Here we try the heads of one layer at a time. LAWFUL's τ-consistent circuits need M0 > 0. If M0 ≤ 0 at this checkpoint, read the columns relatively: a smaller `d_suf` means more of the model's own response is routed through that layer.

In [ ]:
allc = R.components(em.cfg.depth, em.cfg.heads)
rows = []
for L in ([5] if FAST else range(em.cfg.depth)):
    comps = [("a", L, h) for h in range(em.cfg.heads)]
    r = R.sufficiency_necessity(em, X, lat, lon, comps, allc, R.Wave(phi0=45),
                                ss=range(2, 8) if FAST else range(2, 15))
    rows.append({"layer": L, **{k: round(v, 3) for k, v in r.items()}})
pd.DataFrame(rows)

## Try this

1. Move the wave: `phi0=30` or `phi0=-50` (Southern Hemisphere), or a narrower `sigma`. Theory says β = 2Ω cos φ / a is weaker poleward. Does the model know?
2. Plant at the jet level only (`levels=(2, 3)`) vs through the depth. Is the model's response more barotropic or baroclinic?
3. LAWFUL's greedy search: grow a set of components one at a time until sufficiency ≥ 0.9·M0, then prune it. Use `R.sufficiency_necessity` in a loop. (exege is getting this as `exege.laws`.)
4. The **position embedding** is the only source of latitude. Swap the position embeddings of two latitude rows (a hook on `hook_pos_embed`) and see whether the westward drift follows.
5. Does the wave's growth rate tell you about baroclinic instability (growth for s ≈ 6–8 in winter)?